# Prepare data

Raw annotation exports -> gold standard -> splits. Needs `cfg.ANNOTATION_DIR`, which is not part of the repo.

In [ ]:
import sys
sys.path.insert(0, "..")

from embodiment import config as cfg

import os
from pathlib import Path

import pandas as pd

ANN = cfg.ANNOTATION_DIR
MASTER = ANN / "Data" / "en_embodiment_annotations_master.csv"
LABEL_STUDIO = ANN / "Data" / (
    "embodiment_random_samples_full_corpus-at-2026-06-02-12-30-48747f45.csv")
APP = ANN / "app" / "annotations.csv"
BROWSE = ANN / "app" / "browse_annotations.csv"

# Identifiers as they appear in the private exports. Annotator A ran the
# Label Studio and browse-review rounds, annotator B the app round.
ANNOTATOR_A_EMAIL = os.environ.get("ANNOTATOR_A_EMAIL", "a@example.org")
ANNOTATOR_A_NAME = os.environ.get("ANNOTATOR_A_NAME", "A")
ANNOTATOR_B_NAME = os.environ.get("ANNOTATOR_B_NAME", "B")

## Label normalisation

In [ ]:
# Every variant seen in the exports, mapped to the canonical name. Three labels
# from earlier drafts of the scheme are folded into no_rel.
LABEL_NORM = {
    "NO_REL": "no_rel", "CHARACTER_MOVE": "character_move",
    "HAPTIC_MATERIAL": "haptic_material",
    "CHARACTER_FELTNESS": "character_feltness",
    "SIGHT_HUMAN": "sight_human", "BOOK_PARATEXT": "book_paratext",
    "HAPTIC_HUMAN_ANIMAL": "haptic_human_animal",
    "SIGHT_MATERIAL": "sight_material",
    "CHARACTER_STATIC": "character_static",
    "DESCRIPTION_FELTNESS": "description_feltness",
    "HUMAN_COGNITION": "character_cognition",
    "facial_expression": "facial_expr",
    "sents_seg_error": "segmentation_error",
    "sight_mmaterial": "sight_material",
}


def is_multi(val):
    s = str(val).strip()
    return s.startswith("[") and s.endswith("]")


def normalise(val):
    """Canonical label, or None for multi-label cells and unknown values."""
    if not isinstance(val, str) or is_multi(val):
        return None
    norm = LABEL_NORM.get(val.strip(), val.strip().lower())
    return "no_rel" if norm in cfg.RETIRED else norm

## Sources

In [ ]:
def frame(df, filename, sentence_id, text, label, source):
    out = pd.DataFrame({
        "filename": df[filename].values, "sentence_id": df[sentence_id].values,
        "text": df[text].values, "label": df[label].apply(normalise).values,
        "source": source,
    })
    return out.dropna(subset=["label"])


ls = pd.read_csv(LABEL_STUDIO)
ls = ls[ls["annotator"] == ANNOTATOR_A_EMAIL]
ls_out = frame(ls, "source_file", "sentence_id", "text", "embodiment",
               "label_studio_a")

app = pd.read_csv(APP)
app = app[app["annotator"] == ANNOTATOR_B_NAME]
app_out = frame(app, "filename", "sentence_id", "text", "label", "app_b")

browse = pd.read_csv(BROWSE)
browse = browse[(browse["annotator"] == ANNOTATOR_A_NAME)
                & (browse["label_set"] == "embodiment")
                & ~browse["feedback"].isin(["correct", "incorrect"])]
browse_out = frame(browse, "filename", "sentence_id", "text", "feedback",
                   "browse_a")

master = pd.read_csv(MASTER)
master_out = frame(master, "filename", "sentence_id", "text", "space_type",
                   "master")

for name, part in [("label_studio_a", ls_out), ("app_b", app_out),
                   ("browse_a", browse_out), ("master", master_out)]:
    print(f"{name:14s} {len(part):5d}")

## Combine

In [ ]:
# Concatenation order is the precedence order: the first occurrence of a
# sentence wins, so a later re-annotation of the same sentence is dropped.
combined = pd.concat([ls_out, app_out, browse_out, master_out],
                     ignore_index=True)
before = len(combined)
combined = combined.drop_duplicates(subset=["filename", "sentence_id"],
                                    keep="first")
print(f"{before} -> {len(combined)} after dedup")
print(combined["label"].value_counts().to_string())

## Downsample no_rel

In [ ]:
counts = combined["label"].value_counts()
no_rel = combined[combined["label"] == "no_rel"]
rest = combined[combined["label"] != "no_rel"]
target = min(len(no_rel), counts[counts.index != "no_rel"].iloc[0] * 2)

balanced = pd.concat([rest, no_rel.sample(n=target, random_state=cfg.SEED)],
                     ignore_index=True)
balanced = balanced.sample(frac=1, random_state=cfg.SEED).reset_index(drop=True)
print(f"no_rel {len(no_rel)} -> {target};  total {len(balanced)}")

## Apply corrections

In [ ]:
# label_corrections.csv is append-only; the newest row for a sentence wins.
corr = pd.read_csv(cfg.CORRECTIONS).drop_duplicates(
    ["filename", "sentence_id"], keep="last")
assert corr["new_label"].isin(cfg.LABELS).all()

fix = corr.set_index(["filename", "sentence_id"])["new_label"]
idx = pd.MultiIndex.from_frame(balanced[["filename", "sentence_id"]])
mask = idx.isin(fix.index)
balanced.loc[mask, "label"] = idx[mask].map(fix)
print(f"applied {mask.sum()} of {len(corr)} logged corrections")

gold = balanced[balanced["label"].isin(cfg.LABELS)].reset_index(drop=True)
gold.to_csv(cfg.GOLD, index=False)
print(f"gold: {len(gold)} sentences -> {cfg.GOLD.name}")

## Split

In [ ]:
from sklearn.model_selection import train_test_split

enc = gold.assign(label=gold["label"].map(cfg.LABEL2ID))
train, rest = train_test_split(enc, test_size=0.30, random_state=cfg.SEED,
                               stratify=enc["label"])
dev, test = train_test_split(rest, test_size=0.50, random_state=cfg.SEED,
                             stratify=rest["label"])

for frame_, path in [(train, cfg.TRAIN), (dev, cfg.DEV), (test, cfg.TEST)]:
    frame_.to_csv(path, index=False)
print(f"train {len(train)} / dev {len(dev)} / test {len(test)}")